# Usages

## Overview

- This page provides a general pipeline for analyzing Lipidomics (LiP) data using the `lipana` package.
- The example data used here is a truncated search report from DIA-NN. This dataset includes three experiment conditions, each with three replicates, and only 1000 proteins are retained.
- The example files are located in the `"path_to/LiPAna/example_data"` directory.

In [6]:
import gzip
import shutil
from pathlib import Path

workspace = Path(".").resolve().parents[1].joinpath("example_data")
print("current workspace:", str(workspace))
# Unzip gzipped files
for file in workspace.glob("*.gz"):
    with gzip.open(file, "rb") as f_in:
        with open(file.with_suffix(""), "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)


current workspace: /home/lishsh/LiPAna_new_version/doc/example_data


In [9]:
import sys

sys.path.append(str(workspace.parent.parent))
import lipana

In [10]:
from time import sleep

import numpy as np
import polars as pl

## Define the Experiment Layout

To annotate the expected report, it is essential to establish an experiment layout. Two methods for accomplishing this are introduced below:

1. Loading from a Text File: You can load a text file containing the experiment layout. An example of such a file might look like this ("replicate" column can be omitted):

|  run | condition | replicate |
| ---: | --------: | --------: |
| run1 |      ctrl |         1 |
| run2 |      ctrl |         2 |
| run3 |      ctrl |         3 |
| run4 |      exp1 |         1 |
| run5 |      exp1 |         2 |
| run6 |      exp1 |         3 |
| run7 |      exp2 |         1 |
| run8 |      exp2 |         2 |

In [11]:
import lipana

from pathlib import Path
workspace = Path("/home/lishsh/LiPAna_new_version/example_data")

exp_layout = lipana.ExperimentLayout.from_file(workspace.joinpath("experiment_layout.txt"))
print("The input file is:")
exp_layout.exp_df

The input file is:


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2_A""","""H2_Y100""",1
"""20240331_YSJ-HY_2_B""","""H2_Y100""",2
"""20240404_YSJ-HY_2_C""","""H2_Y100""",3
"""20240331_YSJ-HY_5_A""","""H5_Y100""",1
"""20240331_YSJ-HY_5_B""","""H5_Y100""",2
"""20240404_YSJ-HY_5_C""","""H5_Y100""",3
"""20240331_YSJ-HY_30_A""","""H30_Y100""",1
"""20240331_YSJ-HY_30_B""","""H30_Y100""",2
"""20240404_YSJ-HY_30_C""","""H30_Y100""",3


2. Initializing with a Run-to-Condition Map: Alternatively, you can define the experiment layout by passing a dictionary mapping runs to their respective conditions. This approach is useful if you need to programmatically create or adjust the layout based on specific experiment parameters without relying on an external file

In [51]:
import lipana

exp_layout = lipana.ExperimentLayout.from_run_to_condition_map(
    {
        "20240331_YSJ-HY_2_A": "H2_Y100",
        "20240331_YSJ-HY_2_B": "H2_Y100",
        "20240404_YSJ-HY_2_C_": "H2_Y100",
        "20240331_YSJ-HY_5_A": "H5_Y100",
        "20240331_YSJ-HY_5_B": "H5_Y100",
        "20240404_YSJ-HY_5_C": "H5_Y100",
        "20240331_YSJ-HY_30_A": "H30_Y100",
        "20240331_YSJ-HY_30_B": "H30_Y100",
        "20240404_YSJ-HY_30_C": "H30_Y100",
    }
)

print("This results in the same experiment layout as the previous example, because replicate is incremented from 1")
exp_layout.exp_df

This results in the same experiment layout as the previous example, because replicate is incremented from 1


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2_A""","""H2_Y100""",1
"""20240331_YSJ-HY_2_B""","""H2_Y100""",2
"""20240404_YSJ-HY_2_C_""","""H2_Y100""",3
"""20240331_YSJ-HY_5_A""","""H5_Y100""",1
"""20240331_YSJ-HY_5_B""","""H5_Y100""",2
"""20240404_YSJ-HY_5_C""","""H5_Y100""",3
"""20240331_YSJ-HY_30_A""","""H30_Y100""",1
"""20240331_YSJ-HY_30_B""","""H30_Y100""",2
"""20240404_YSJ-HY_30_C""","""H30_Y100""",3


## Load a search report

- This step involves loading and annotating the search report
  - A DIA-NN report can originate from either running DIA-NN standalone or using FragPipe, which integrates DIA-NN for DIA data processing
  - For Spectronaut reports, certain columns are required, and a template for report settings can be generated using: `lipana.report.export_sn_report_setting("path_to_store_the_setting_file.rs")`

### First Load FASTA File(s)

- Before annotating the report, it is essential to load and parse FASTA files for species and sequence information
- `lipana.parse_fasta` will read one or more FASTA files and return a `ParsedFasta` object
  - `fasta_path` can be a path points to a FASTA or a list of paths (`fasta_path=some_path` or `fasta_path=[path1, path2]`)
  - `contam_fasta_path`: this parameter will be useful to strictly exclude potential contaminants. The species of any protein entries in the given file will be annotated as "Contam" instead of its original species, including those presented in `fasta_path`
  - `contaminations`: a list of additional protein entries can be given here, and this parameter can also work alone if `contam_fasta_path` is `None`; this parameter can also be used to define all the entries that from a certain species as contaminants, by providing the species name(s) in the list
- For title parsing
  - Parameter `fasta_title_regex` is `None` by default, and the title will be parsed as the UniProt format ">sp|protein|name_species ...", which will extract "protein" and "species"
  - To set a customized regex for parsing titles, this parameter can be `str` or a list of `str`
  - For example
    - `fasta_title_regex=None` -> UniProt title
    - `fasta_title_regex=r">[^|\s]+?\|(?P<protein>[^|\s]+?)\|[^\s]+?_(?P<species>[^\s]+)[$\s].*"` -> UniProt title but by regex
    - `fasta_title_regex=r">(?P<protein>[^\ ]+).+?Tax_Id=(?P<species>[^\ ]+).*?"` -> A format like ">Q32MB2 TREMBL:Q32MB2 Tax_Id=9606 Gene_Symbol=KRT73"
    - `fasta_title_regex=[regex1, regex2]` -> Try to parse title sequentially until both "protein" and "species" are found. If "protein" can not be found by any regex, will raise error. If "species" can not be found, it will be "unknown", and "protein" will be the first matched one.

In [12]:
parsed_fasta = lipana.parse_fasta(
    fasta_path=workspace.joinpath("human_yeast_contam.fasta"),
    contam_fasta_path=workspace.joinpath("contam.fasta"),
    contaminations=None,
    fasta_title_regex=None,  # Set one or more regex if the title is not UniProt format or there are mixed formats in the FASTA files
    gen_species_to_concat_seqs=True,  # This parameter is useful to annotate species for peptides via a whole protein sequence map, but is time-consuming
    workspace=workspace,
    resume=True,
    write_parsed_fasta=True,
)

### Load and annotate a search report

- Now, load and annotate the search report using the previously set up `ExperimentLayout` and `ParsedFasta`

In [13]:
# The load_search_report methods for DIA-NN and Spectronaut share parameters:
# lipana.DIANNReport.load_search_report(...)
# lipana.SpectronautReport.load_search_report(...)

# Example loading a DIA-NN search report, showing a subset of available parameters
report = lipana.DIANNReport.load_search_report(
    workspace.joinpath("example_diann_report.tsv"),
    exp_layout=exp_layout,
    parsed_fasta=parsed_fasta,
    do_species_annotation=True,  # Set this to True if the experiment includes multiple species
    pre_annotation_filter=(
        (pl.col("Q.Value") < 0.01)
        & (pl.col("Lib.PG.Q.Value") < 0.01)
        & (pl.col("Protein.Group").is_not_null())
        & (pl.col("Precursor.Quantity").is_not_null())
        & (pl.col("Precursor.Quantity") > 1.1)
    ),  # The filters defined here will be applied to the report after loading immediately, and here the 5 rules are default for DIA-NN (if this parameter is not given, these rules will also be applied); for Spectronaut, they are (
    #     (pl.col("PG.ProteinGroups").is_not_null())
    #     & (pl.col("FG.Quantity").is_not_null())
    #     & (pl.col("FG.Quantity") > 1.1)
    # )
    post_annotation_filter=None,  # This parameter can be used to filter the report after annotation
    restricted_cut_sites=("K", "R"),
    expand_to_cut_site_level=True,  # This will make each cut site one row, which means each precursor row will be duplicated
    # modification_map=None,  # This parameter can be either None or a map from software used modification to UniMod. By default, DIA-NN report has None because it use UniMod as default; for `SpectronautReport`, this parameter has a default value as {
    #     "[Acetyl (Protein N-term)]": "(UniMod:1)",
    #     "[Carbamidomethyl (C)]": "(UniMod:4)",
    #     "[Oxidation (M)]": "(UniMod:35)",
    # }
    resume=True,
)
# The variable `report` is now a `SearchReport` object, and basic annotations have been done
report

Search report object with main report shape: (113057, 81)
Number of quantification data: 0
Number of statistics results: 0
Workspace: /home/lishsh/LiPAna_new_version/example_data

## Preparation of Quantification Matrices

- A newly created `SearchReport` object initially contains only a preprocessed search report and associated metadata, and in this section, some quantification matrices will be added to it
- The quantification process can simply involve pivoting the report or aggregating data using methods like topN or maxLFQ

### Use raw reported quantity values

- This approach converts the report into a quantification matrix by using the raw quantity values reported by the software as matrix cells. It is particularly useful for generating precursor-level or protein group-level matrices, since most software will report the quantity values on these two levels
- Additionally, this step demonstrates how to manually attach a customized quantification report to the main report using the `attach_quant_data` method

In [14]:
quant_data = report.attach_quant_data(
    lipana.convert_long_report_to_wide(
        report,
        index_col="precursor",
        column_col="run",
        value_col="precursor_quantity",
        do_log_scale=2,
        pl_filter=(
            ~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)
        ),  # This parameter filters out those peptides mapped to multiple species
        do_unique=True,
    ),
    # These two parameters will be the name of this quantification report, and can be used to retrieve it from the main report object
    # Though they can be named as any strings, it is recommended to use the same name as the entry level and the actually used quantification method
    entry_level="precursor",
    quant_method="precursor_quantity",
)

# To retrieve this object from the main report object, use the following command
quant_data = report.get_quant_data("precursor", "precursor_quantity")


### maxLFQ and topN Methods

- Both maxLFQ and topN methods utilize the same interface through the `SearchReport.construct_and_attach_quant_data` method, featuring convenient designs to handle more complex scenarios
- The following examples will concentrate on maxLFQ; topN can be configured by setting parameters like `method="top3"` or `method="top1"`, etc.


#### Peptide matrix from precursor MS2 via maxLFQ

- This example will do maxLFQ on precursor quantities to get a peptide level matrix

In [15]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_prec",  # This parameter will be the name of quantification method for this quantification report, and can be used to retrieve it from the main report object
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",  # This parameter not only define the primary (target) entry level, but also the first name to retrieve the generated matrix from the main report object
    low_level_entry_col="precursor",
    base_quant_col="precursor_quantity",
    require_expansion=False,
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmp2vwk1u_d.txt
Removing low intensities...

Output to /tmp/tmp2vwk1u_d.txt-iq_output.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp2vwk1u_d.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 55844
# quantitative values after filtering = 55844

# samples  = 9
# proteins = 9577
nrow = 55844, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
20%
25%
30%
35%
40%
45%
50%
55%
60%
66%
71%
76%
81%
87%
92%
97%
Completed.


#### Peptide matrix from prec MS1 and prec MS2 via maxLFQ

- The quantities of base level entries can also be from multiple sources
- The following example will use maxLFQ to aggregate quantities from two levels: precursor MS1 and precursor MS2

In [16]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col=("precursor", "precursor"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1"),
    require_expansion=(False, False),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmpllcp0prr.txt
Removing low intensities...




Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmpllcp0prr.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 111685
# quantitative values after filtering = 111685

# samples  = 9
# proteins = 9577
nrow = 111685, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
21%
27%
32%
38%
43%
49%
55%
61%
66%
71%
77%
82%
87%
92%
98%
Completed.


Output to /tmp/tmpllcp0prr.txt-iq_output.txt


#### Peptide matrix from prec MS1, prec MS2, and fragments, via maxLFQ

- The following example shows a more complicated case, which uses maxLFQ to aggregate quantities from three levels: precursor MS1, precursor MS2, and fragments (the fragment quantity values in the DIA-NN tsv report is in one table cell joined by ";" for each row)


In [17]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_ms1ms2frag",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col=("precursor", "precursor", "Fragment.Info"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1", "Fragment.Quant.Raw"),
    require_expansion=(False, False, ";"),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmp6vuvahnl.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp6vuvahnl.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 701484
# quantitative values after filtering = 701484

# samples  = 9
# proteins = 9577


Removing low intensities...



nrow = 701484, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
22%
28%
33%
38%
44%
49%
55%
61%
66%
71%
77%
84%
90%
95%
Completed.


Output to /tmp/tmp6vuvahnl.txt-iq_output.txt


#### Non-restricted cut site from precursor MS2+MS1

- Sometimes, the quantity values should be aggregated from a filtered report
- The following in-one-step example will generate a cut site level matrix, where the cut sites are only non-restricted
  - Whether a cut site is restricted has been annotated via the parameter `restricted_cut_sites` in method `load_search_report` (in this doc, it's `lipana.DIANNReport.load_search_report(..., restricted_cut_sites=("K", "R"), ...)`)
  - Besides the filtering for site annotation `(~pl.col("cut_site_is_restricted"))`, there is also a filter  for peptide, because the C-term of a cut site can be the second residue of a protein, which would be surrounded by first "M" and the third any residue. Here the cut site will be annotated as "non-restricted" if the third residue is K/R, but it's hard to say if the cleavage between protein N-term "M" and the second residue is caused during translation or by a non-restricted enzyme. Further restraining peptide to be semi-specific will address this problem by filtering out such peptides at the protein N-term.

In [18]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="nonrestricted_cut_site_by_maxlfq_from_prec",
    method="maxlfq",
    filter_condition=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
        & (~pl.col("cut_site_is_restricted"))
    ),
    run_col="run",
    primary_entry_col="cut_site",
    low_level_entry_col=("precursor", "precursor"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1"),
    require_expansion=(False, False),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)


Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmpio30a3hr.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 52579
# quantitative values after filtering = 52579

# samples  = 9
# proteins = 4116
nrow = 52579, # proteins = 4116, # samples = 9
Using 95 threads...
0%
5%
10%
15%
21%
26%
31%
37%
43%
54%
59%
65%
70%
76%
81%
86%
92%
98%
Completed.


MaxLFQ estimation via iq for /tmp/tmpio30a3hr.txt
Removing low intensities...

Output to /tmp/tmpio30a3hr.txt-iq_output.txt


### Utilities on quantification report

- The quantification matrix generated from the above sections is actually an object `EntryQuantificationReport`, which has many handful functions

In [19]:
# If the generated object is not explicitly assigned to a variable, can use the following way to get from main report object
quant_data = report.get_quant_data("precursor", "precursor_quantity")

# The following three methods will attach new columns to the dataframe stored in the quantification report object
quant_data.calc_cv(cond="all", min_reps=2, temp_reverse_log_scale=2, new_colname_pattern="{cond}_cv_{min_reps}reps")
quant_data.count_detected_replicates(new_colname_pattern="{cond}_detected_reps")
quant_data.calc_ratio(
    base_cond="H5_Y100",
    is_log=True,
    temp_reverse_log_scale=2,
    div_method="agg_and_divide",
    agg_method="mean",
    new_colname_pattern="ratio_{cond1}_to_{cond2}",
)

precursor,20240331_YSJ-HY_30_A,20240404_YSJ-HY_2_C,20240404_YSJ-HY_5_C,20240331_YSJ-HY_2_B,20240331_YSJ-HY_30_B,20240331_YSJ-HY_5_B,20240404_YSJ-HY_30_C,20240331_YSJ-HY_5_A,20240331_YSJ-HY_2_A,H2_Y100_cv_2reps,H5_Y100_cv_2reps,H30_Y100_cv_2reps,H2_Y100_detected_reps,H5_Y100_detected_reps,H30_Y100_detected_reps,ratio_H2_Y100_to_H5_Y100,ratio_H30_Y100_to_H5_Y100
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32
"""AQNIDYDKLIK/2""",14.837064,14.921881,14.924777,15.381221,15.061318,15.121512,14.975474,15.308312,15.524149,20.617542,13.241808,7.767026,3,3,3,0.171129,-0.165786
"""GAGKPNSSAVAILETF/2""",15.141305,14.765122,15.500525,15.360956,15.160276,14.896846,15.426919,14.943264,15.108235,20.300888,24.576078,11.393847,3,3,3,-0.042147,0.108213
"""SNTTATKEEFDDQLK/3""",13.460197,14.085488,14.036002,13.630456,null,13.934687,13.724004,14.05909,13.919411,15.581356,4.532853,12.894028,3,3,2,-0.120421,-0.412809
"""TGDVEDSTVLK/2""",17.339526,13.270021,14.576235,12.29316,17.141684,14.705182,16.93105,14.455681,13.062735,32.1366,8.65313,14.092543,3,3,3,-1.650101,2.5644
"""RSTLDPVEK/2""",20.203632,20.667521,20.674458,20.389841,20.311899,20.375923,20.568141,20.266167,20.192991,16.69326,15.032695,13.251674,3,3,3,-0.019343,-0.079913
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VAVAIPNRPPDAVLTDTTSLNQAALYR/4""",null,null,null,null,null,null,16.226385,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""YGGPNHIVGSPFK/3""",13.951588,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""VKEQDYLC(UniMod:4)HVYVR/3""",null,null,null,null,null,null,12.952363,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN


- Because main report and quantification report are related, the annotation and filtering of quantification report can be done by borrowing data from the main report
- The following method will return a new dataframe instead of updating the dataframe stored in the object

In [20]:
report.get_quant_data(
    entry_name="precursor",
    quant_name="precursor_quantity",
    main_df_entry_filter=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
    ),
    quant_df_filter=None,
    annotation_cols="protein_group",
)

precursor,20240331_YSJ-HY_30_A,20240404_YSJ-HY_2_C,20240404_YSJ-HY_5_C,20240331_YSJ-HY_2_B,20240331_YSJ-HY_30_B,20240331_YSJ-HY_5_B,20240404_YSJ-HY_30_C,20240331_YSJ-HY_5_A,20240331_YSJ-HY_2_A,H2_Y100_cv_2reps,H5_Y100_cv_2reps,H30_Y100_cv_2reps,H2_Y100_detected_reps,H5_Y100_detected_reps,H30_Y100_detected_reps,ratio_H2_Y100_to_H5_Y100,ratio_H30_Y100_to_H5_Y100,protein_group
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32,str
"""GAGKPNSSAVAILETF/2""",15.141305,14.765122,15.500525,15.360956,15.160276,14.896846,15.426919,14.943264,15.108235,20.300888,24.576078,11.393847,3,3,3,-0.042147,0.108213,"""P40202"""
"""SNTTATKEEFDDQLK/3""",13.460197,14.085488,14.036002,13.630456,null,13.934687,13.724004,14.05909,13.919411,15.581356,4.532853,12.894028,3,3,2,-0.120421,-0.412809,"""P10592"""
"""RSTLDPVEK/2""",20.203632,20.667521,20.674458,20.389841,20.311899,20.375923,20.568141,20.266167,20.192991,16.69326,15.032695,13.251674,3,3,3,-0.019343,-0.079913,"""P10592"""
"""VKPVGIVVPNLGHL/3""",11.565403,10.710749,12.065379,11.01836,11.664795,null,11.900337,11.069642,null,15.020044,46.954628,12.159408,2,2,3,-0.779027,0.065343,"""P47912"""
"""YDTDVFTPLFER/2""",null,10.493862,11.40034,11.426908,11.566868,10.910385,12.615703,9.999928,null,44.201256,44.603138,49.255928,2,3,2,0.155056,1.305091,"""P40825"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""LTPTQGSYTFDPVTK/2""",12.932636,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""Q9Y2T2"""
"""TVDKDQSIAPK/2""",13.72056,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""P04844"""
"""TTPSFVGFTD/2""",null,12.659182,null,null,null,null,null,null,null,NaN,NaN,NaN,1,0,0,NaN,NaN,"""P10592"""


In [24]:
quant_data = report.get_quant_data("precursor", "precursor_quantity")
quant_data.attach_annotation_via_entry(
    "protein_group", persist=True
)  # Set `persistent` to True will update the dataframe stored in the object
quant_data.filter_entry_by_main_report(
    main_report_filter=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
    & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
)


precursor,20240331_YSJ-HY_30_A,20240404_YSJ-HY_2_C,20240404_YSJ-HY_5_C,20240331_YSJ-HY_2_B,20240331_YSJ-HY_30_B,20240331_YSJ-HY_5_B,20240404_YSJ-HY_30_C,20240331_YSJ-HY_5_A,20240331_YSJ-HY_2_A,H2_Y100_cv_2reps,H5_Y100_cv_2reps,H30_Y100_cv_2reps,H2_Y100_detected_reps,H5_Y100_detected_reps,H30_Y100_detected_reps,ratio_H2_Y100_to_H5_Y100,ratio_H30_Y100_to_H5_Y100,protein_group
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32,str
"""GAGKPNSSAVAILETF/2""",15.141305,14.765122,15.500525,15.360956,15.160276,14.896846,15.426919,14.943264,15.108235,20.300888,24.576078,11.393847,3,3,3,-0.042147,0.108213,"""P40202"""
"""SNTTATKEEFDDQLK/3""",13.460197,14.085488,14.036002,13.630456,null,13.934687,13.724004,14.05909,13.919411,15.581356,4.532853,12.894028,3,3,2,-0.120421,-0.412809,"""P10592"""
"""RSTLDPVEK/2""",20.203632,20.667521,20.674458,20.389841,20.311899,20.375923,20.568141,20.266167,20.192991,16.69326,15.032695,13.251674,3,3,3,-0.019343,-0.079913,"""P10592"""
"""VKPVGIVVPNLGHL/3""",11.565403,10.710749,12.065379,11.01836,11.664795,null,11.900337,11.069642,null,15.020044,46.954628,12.159408,2,2,3,-0.779027,0.065343,"""P47912"""
"""YDTDVFTPLFER/2""",null,10.493862,11.40034,11.426908,11.566868,10.910385,12.615703,9.999928,null,44.201256,44.603138,49.255928,2,3,2,0.155056,1.305091,"""P40825"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""LTPTQGSYTFDPVTK/2""",12.932636,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""Q9Y2T2"""
"""TVDKDQSIAPK/2""",13.72056,null,null,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""P04844"""
"""TTPSFVGFTD/2""",null,12.659182,null,null,null,null,null,null,null,NaN,NaN,NaN,1,0,0,NaN,NaN,"""P10592"""


## Statistics

- The `lipana` package provides several built-in functions that users can chain together to build custom statistical pipelines. For convenience, the package also includes some wrapper functions that provides pre-defined pipeline.
- This section demonstrates how to use these pre-defined pipelines for performing statistical tests.
- Note (missing value): The `lipana.do_stats_pipe_direct` function includes an optional `mv_config` parameter, which defaults to `None`.
  - Default Behavior (`mv_config=None`): If `mv_config` is not specified or is set to `None`, the input data is filtered first based on the `min_rep_count` threshold before any statistical testing occurs.
  - Using Missing Value Handlers: You can set `mv_config` to an instance (or a list of instances) of `AbstractPairwiseMissingValueHandler`. Examples include: `FullEmptyFillingMissingValueHandler(min_rep_count=3)` and `NormalDistSamplingMissingValueHandler(min_rep_count=3, do_imputation_for_rep_range=(1, 2), log_scale_minmax_diff=1)`. When a handler is provided, missing value processing is performed first, and the `min_rep_count` requirement is checked afterward.
  - Example Handler Usage: The examples below utilize `FullEmptyFillingMissingValueHandler(min_rep_count=3)`. This handler imputes a value of 0 (on the log scale) for entries where one condition lacks detected replicates, provided the other condition in the pair has at least 3 detected replicates (`>= min_rep_count`).
  - Disabling Imputation: To disable specific missing value filling or imputation while still potentially using the handler structure, set `mv_config` to `lipana.NullMissingValueHandler()`. Filtering will still be based on `min_rep_count` after this (null) handling step.
- Note (aggregation): The `lipana.do_stats_pipe_agg` function uses the `pipeline` parameter to control how results are aggregated from a base level (e.g., peptides) to a target level (e.g., proteins or cut sites). Specify one of the following options for the `pipeline` parameter:
  - `"sel_min_p"`: Selects the single base-level entry with the minimum p-value for each target-level entry.
  - `"sel_min2_p"`, `"sel_min3_p"`, ...: Selects the top 2, 3, etc., base-level entries with the lowest p-values for each target-level entry.
  - `"combine_p"`: Combines p-values from all associated base-level entries for each target-level entry using Fisher's method. The reported fold change (FC) will be the median FC among the combined base-level entries.
  - Pipelines with `_direction_check` Suffix (e.g., `"sel_min_p_direction_check"`, `"combine_p_direction_check"`): These pipelines first evaluate the directionality (sign of the t-statistic in the predefined pipelines) of fold changes for all base entries mapping to a target entry. They determine if there is a majority direction (positive or negative). Only the base entries matching this majority direction are considered for the subsequent selection or combination step.
- Note (significant selection): After hypothesis testing (and potential aggregation), the `lipana.stats.do_significant_selection` function can be used to annotate whether a target entry is significant, based on specified rules.
  - Generally, you only need to set the `rules` parameter to one or a list of `lipana.stats.SignificantRule` instances, which define the thresholds and pre-filtering conditions used to determine if an entry is significant.
  - To meet the potential requirement that more than one significant detection at the base level is needed to support the significance of a target entry, `requires_n_passed` can be set to a value greater than 1. In this case, `target` will be used as the grouping key to count significant base entries per target.


### Example 1: Test on Fully-Specific Stripped Peptides

This example performs hypothesis tests directly on the quantification data for fully-specific stripped peptides. The results remain at the peptide level.

#### Performing Hypothesis Tests on Fully-Specific Stripped Peptides

- First, we select the fully-specific peptides from the complete search report, and then perform hypothesis tests on this subset.


In [25]:
design = lipana.ComparisonDesign(exp_layout, (("H2_Y100", "H5_Y100"), ("H30_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = (pl.col("peptide_enzymatic_specificity") == "fully_specific") & (
    pl.col("mapped_species_from_peptide").is_in(("HUMAN", "YEAST"))
)

stats_fully_pep = lipana.stats.pipe.do_stats_pipe_direct(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    design=design,
    entry_level="stripped_peptide",
    mv_config=lipana.FullEmptyFillingMissingValueHandler(3),
    min_rep_count=3,
    annotation_df=report.df.filter(filters),
    group_entry_level="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    annotation_cols=[
        "peptide_start_position",
        "peptide_end_position",
        "prev_aa",
        "next_aa",
        "peptide_n_term_aa",
        "peptide_c_term_aa",
        "nterm_enzymatic_specificity",
        "cterm_enzymatic_specificity",
        "peptide_enzymatic_specificity",
        "n_cut_site",
        "c_cut_site",
        "mapped_species_from_peptide",
    ],
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats_fully_pep.shape}):")
stats_fully_pep.head(5)

design.pairwise_comparisons = (('H2_Y100', 'H5_Y100'), ('H30_Y100', 'H5_Y100'))


Quantification file: /tmp/tmp5a6zgdi1.txt
All defined comparison pairs: c("H2_Y100", "H5_Y100")
Available comparison pairs: c("H2_Y100", "H5_Y100")
Comparing H2_Y100 vs H5_Y100
Output to /tmp/tmp5a6zgdi1.txt-limma_output.txt
Quantification file: /tmp/tmpxyatj5qn.txt
All defined comparison pairs: c("H30_Y100", "H5_Y100")
Available comparison pairs: c("H30_Y100", "H5_Y100")
Comparing H30_Y100 vs H5_Y100
Output to /tmp/tmpxyatj5qn.txt-limma_output.txt


First 5 rows of the result (total size: (10126, 28)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,mapped_species_from_peptide
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,str
"""RIEYIEAR""",-14.269712,7.134856,-150.890513,1.1007e-12,3.6068e-10,15.988599,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.6068e-10,"""Q8WUW1""",1.1007e-12,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""","""HUMAN"""
"""AGAFEHLPSLR""",-13.485006,6.742503,-145.831244,1.3713e-12,3.6068e-10,15.943529,"""H2_Y100_vs_H5_Y100""",-13.485344,0,3,"""full_empty""",true,3.6068e-10,"""Q13641""",1.3713e-12,135,145,"""R""","""Q""","""A""","""R""","""restricted""","""restricted""","""fully_specific""","""Q13641-134_135""","""Q13641-145_146""","""HUMAN"""
"""QMADTGKLNTLLQR""",-13.452959,6.72648,-145.690779,1.3799e-12,3.6068e-10,15.942218,"""H2_Y100_vs_H5_Y100""",-13.453273,0,3,"""full_empty""",true,3.6068e-10,"""Q00839""",2.0698e-11,545,558,"""K""","""A""","""Q""","""R""","""restricted""","""restricted""","""fully_specific""","""Q00839-544_545""","""Q00839-558_559""","""HUMAN"""
"""LQDLQGEKDALHSEK""",-13.04591,6.522955,-140.910685,1.7109e-12,3.6068e-10,15.895561,"""H2_Y100_vs_H5_Y100""",-13.046274,0,3,"""full_empty""",true,3.6068e-10,"""O60610""",1.1977e-11,513,527,"""K""","""Q""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""O60610-512_513""","""O60610-527_528""","""HUMAN"""
"""LTEELKEQMK""",-12.956682,6.478341,-135.438894,2.2086e-12,3.6068e-10,15.836811,"""H2_Y100_vs_H5_Y100""",-12.957698,0,3,"""full_empty""",true,3.6068e-10,"""Q14683""",3.2154e-12,681,690,"""R""","""A""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""Q14683-680_681""","""Q14683-690_691""","""HUMAN"""


#### Selecting Significant Detections (General Purpose)

- In a typical experiment, significant detections might be defined as having a fold change (FC) greater than 1.5 or less than 1/1.5, combined with an adjusted p-value below 0.05.
- This selection can be achieved using the following commands.


In [26]:
stats_fully_pep = lipana.stats.do_significant_selection(
    stats_fully_pep,
    rules=lipana.stats.SignificantRule(
        greater_than=None,
        less_than=("adj_pvalue_group_wise", 0.05),
        gt_value_or_lt_negate=(
            "log2_fc_limma",
            np.log2(1.5),
        ),  # note the values defined in this paramter will be expand to (`> value` or `< -value`)
        equal_to=None,
        filter_condition=None,
    ),
    requires_n_passed=1,
    target=None,
    mark_col="significant",
)
filtered = stats_fully_pep.filter(pl.col("significant"))
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (2228, 29)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,mapped_species_from_peptide,significant
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,str,bool
"""RIEYIEAR""",-14.269712,7.134856,-150.890513,1.1007e-12,3.6068e-10,15.988599,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.6068e-10,"""Q8WUW1""",1.1007e-12,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""","""HUMAN""",true
"""AGAFEHLPSLR""",-13.485006,6.742503,-145.831244,1.3713e-12,3.6068e-10,15.943529,"""H2_Y100_vs_H5_Y100""",-13.485344,0,3,"""full_empty""",true,3.6068e-10,"""Q13641""",1.3713e-12,135,145,"""R""","""Q""","""A""","""R""","""restricted""","""restricted""","""fully_specific""","""Q13641-134_135""","""Q13641-145_146""","""HUMAN""",true
"""QMADTGKLNTLLQR""",-13.452959,6.72648,-145.690779,1.3799e-12,3.6068e-10,15.942218,"""H2_Y100_vs_H5_Y100""",-13.453273,0,3,"""full_empty""",true,3.6068e-10,"""Q00839""",2.0698e-11,545,558,"""K""","""A""","""Q""","""R""","""restricted""","""restricted""","""fully_specific""","""Q00839-544_545""","""Q00839-558_559""","""HUMAN""",true
"""LQDLQGEKDALHSEK""",-13.04591,6.522955,-140.910685,1.7109e-12,3.6068e-10,15.895561,"""H2_Y100_vs_H5_Y100""",-13.046274,0,3,"""full_empty""",true,3.6068e-10,"""O60610""",1.1977e-11,513,527,"""K""","""Q""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""O60610-512_513""","""O60610-527_528""","""HUMAN""",true
"""LTEELKEQMK""",-12.956682,6.478341,-135.438894,2.2086e-12,3.6068e-10,15.836811,"""H2_Y100_vs_H5_Y100""",-12.957698,0,3,"""full_empty""",true,3.6068e-10,"""Q14683""",3.2154e-12,681,690,"""R""","""A""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""Q14683-680_681""","""Q14683-690_691""","""HUMAN""",true


- Note that the parameter `gt_value_or_lt_negate` will do an "or" combination by chaining two expressions `(> value) | (< -value)`, and this can be more explicitly defined by providing two rules for the `rules` parameter of `do_significant_selection`
- The following command has the same effect as the above one

In [27]:
stats_fully_pep = lipana.stats.do_significant_selection(
    stats_fully_pep,
    rules=[
        lipana.stats.SignificantRule(
            greater_than=("log2_fc_limma", np.log2(1.5)),
            less_than=("adj_pvalue_group_wise", 0.05),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=None,
        ),
        lipana.stats.SignificantRule(
            greater_than=None,
            less_than=(("adj_pvalue_group_wise", 0.05), ("log2_fc_limma", -np.log2(1.5))),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=None,
        ),
    ],
    requires_n_passed=1,
    target=None,
    mark_col="significant_example1",
)
filtered = stats_fully_pep.filter(pl.col("significant_example1"))
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (2228, 30)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,mapped_species_from_peptide,significant,significant_example1
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,str,bool,bool
"""RIEYIEAR""",-14.269712,7.134856,-150.890513,1.1007e-12,3.6068e-10,15.988599,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.6068e-10,"""Q8WUW1""",1.1007e-12,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""","""HUMAN""",true,true
"""AGAFEHLPSLR""",-13.485006,6.742503,-145.831244,1.3713e-12,3.6068e-10,15.943529,"""H2_Y100_vs_H5_Y100""",-13.485344,0,3,"""full_empty""",true,3.6068e-10,"""Q13641""",1.3713e-12,135,145,"""R""","""Q""","""A""","""R""","""restricted""","""restricted""","""fully_specific""","""Q13641-134_135""","""Q13641-145_146""","""HUMAN""",true,true
"""QMADTGKLNTLLQR""",-13.452959,6.72648,-145.690779,1.3799e-12,3.6068e-10,15.942218,"""H2_Y100_vs_H5_Y100""",-13.453273,0,3,"""full_empty""",true,3.6068e-10,"""Q00839""",2.0698e-11,545,558,"""K""","""A""","""Q""","""R""","""restricted""","""restricted""","""fully_specific""","""Q00839-544_545""","""Q00839-558_559""","""HUMAN""",true,true
"""LQDLQGEKDALHSEK""",-13.04591,6.522955,-140.910685,1.7109e-12,3.6068e-10,15.895561,"""H2_Y100_vs_H5_Y100""",-13.046274,0,3,"""full_empty""",true,3.6068e-10,"""O60610""",1.1977e-11,513,527,"""K""","""Q""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""O60610-512_513""","""O60610-527_528""","""HUMAN""",true,true
"""LTEELKEQMK""",-12.956682,6.478341,-135.438894,2.2086e-12,3.6068e-10,15.836811,"""H2_Y100_vs_H5_Y100""",-12.957698,0,3,"""full_empty""",true,3.6068e-10,"""Q14683""",3.2154e-12,681,690,"""R""","""A""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""Q14683-680_681""","""Q14683-690_691""","""HUMAN""",true,true


#### Selecting Significant Detections (Experiment-Specific)

- This example dataset is specific compared to a typical experiment because we have prior knowledge that all human peptides exhibit a fold change of 1/2.5 in the pair `"H2_Y100_vs_H5_Y100"` or 6/1 in the pair `"H30_Y100_vs_H5_Y100"`.
- This example demonstrates how the `do_significant_selection` function handles more customized selection criteria.


In [28]:
stats_fully_pep = lipana.stats.do_significant_selection(
    stats_fully_pep,
    rules=[
        lipana.stats.SignificantRule(
            greater_than=None,
            less_than=(("adj_pvalue_group_wise", 0.05), ("log2_fc_limma", -np.log2(1.5))),
            gt_value_or_lt_negate=None,
            equal_to=None,
            expression_rule=None,  # this parameter allows you to define a custom polars expression to handle more complex conditions
            filter_condition=pl.col("pair").eq("H2_Y100_vs_H5_Y100"),
        ),
        lipana.stats.SignificantRule(
            greater_than=("log2_fc_limma", np.log2(1.5)),
            less_than=("adj_pvalue_group_wise", 0.05),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=pl.col("pair").eq("H30_Y100_vs_H5_Y100"),
        ),
    ],
    requires_n_passed=1,
    target=None,
    mark_col="significant_example2",
)
filtered = stats_fully_pep.filter(pl.col("significant_example2"))
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (2218, 31)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,mapped_species_from_peptide,significant,significant_example1,significant_example2
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,str,bool,bool,bool
"""RIEYIEAR""",-14.269712,7.134856,-150.890513,1.1007e-12,3.6068e-10,15.988599,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.6068e-10,"""Q8WUW1""",1.1007e-12,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""","""HUMAN""",true,true,true
"""AGAFEHLPSLR""",-13.485006,6.742503,-145.831244,1.3713e-12,3.6068e-10,15.943529,"""H2_Y100_vs_H5_Y100""",-13.485344,0,3,"""full_empty""",true,3.6068e-10,"""Q13641""",1.3713e-12,135,145,"""R""","""Q""","""A""","""R""","""restricted""","""restricted""","""fully_specific""","""Q13641-134_135""","""Q13641-145_146""","""HUMAN""",true,true,true
"""QMADTGKLNTLLQR""",-13.452959,6.72648,-145.690779,1.3799e-12,3.6068e-10,15.942218,"""H2_Y100_vs_H5_Y100""",-13.453273,0,3,"""full_empty""",true,3.6068e-10,"""Q00839""",2.0698e-11,545,558,"""K""","""A""","""Q""","""R""","""restricted""","""restricted""","""fully_specific""","""Q00839-544_545""","""Q00839-558_559""","""HUMAN""",true,true,true
"""LQDLQGEKDALHSEK""",-13.04591,6.522955,-140.910685,1.7109e-12,3.6068e-10,15.895561,"""H2_Y100_vs_H5_Y100""",-13.046274,0,3,"""full_empty""",true,3.6068e-10,"""O60610""",1.1977e-11,513,527,"""K""","""Q""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""O60610-512_513""","""O60610-527_528""","""HUMAN""",true,true,true
"""LTEELKEQMK""",-12.956682,6.478341,-135.438894,2.2086e-12,3.6068e-10,15.836811,"""H2_Y100_vs_H5_Y100""",-12.957698,0,3,"""full_empty""",true,3.6068e-10,"""Q14683""",3.2154e-12,681,690,"""R""","""A""","""L""","""K""","""restricted""","""restricted""","""fully_specific""","""Q14683-680_681""","""Q14683-690_691""","""HUMAN""",true,true,true


### Example 2: Test on Semi-Specific Stripped Peptides and Aggregate to Cut Site Level

This example demonstrates a multi-step process:

1. Selects semi-specific peptides from search reports.
2. Performs hypothesis tests on these selected peptides.
3. Annotates the results with non-restricted cut sites.
4. Aggregates the peptide-level p-values and fold changes to the cut site level.

#### Performing Hypothesis Tests on Semi-Specific Stripped Peptides

- First, we select the semi-specific peptides from the complete search report, and then perform hypothesis tests on this subset.


In [29]:
design = lipana.ComparisonDesign(exp_layout, (("H2_Y100", "H5_Y100"), ("H30_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = (pl.col("peptide_enzymatic_specificity") == "semi_specific") & (
    pl.col("mapped_species_from_peptide").is_in(("HUMAN", "YEAST"))
)

stats_semi_pep = lipana.stats.pipe.do_stats_pipe_direct(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    design=design,
    entry_level="stripped_peptide",
    mv_config=lipana.FullEmptyFillingMissingValueHandler(3),
    min_rep_count=3,
    annotation_df=report.df.filter(filters),
    group_entry_level="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    annotation_cols=[
        "peptide_start_position",
        "peptide_end_position",
        "prev_aa",
        "next_aa",
        "peptide_n_term_aa",
        "peptide_c_term_aa",
        "nterm_enzymatic_specificity",
        "cterm_enzymatic_specificity",
        "peptide_enzymatic_specificity",
        "n_cut_site",
        "c_cut_site",
    ],
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats_semi_pep.shape}):")
stats_semi_pep.head(5)

design.pairwise_comparisons = (('H2_Y100', 'H5_Y100'), ('H30_Y100', 'H5_Y100'))


Quantification file: /tmp/tmpkmwf00zm.txt
All defined comparison pairs: c("H2_Y100", "H5_Y100")
Available comparison pairs: c("H2_Y100", "H5_Y100")
Comparing H2_Y100 vs H5_Y100
Output to /tmp/tmpkmwf00zm.txt-limma_output.txt
Quantification file: /tmp/tmpaho7ehmu.txt
All defined comparison pairs: c("H30_Y100", "H5_Y100")
Available comparison pairs: c("H30_Y100", "H5_Y100")
Comparing H30_Y100 vs H5_Y100
Output to /tmp/tmpaho7ehmu.txt-limma_output.txt


First 5 rows of the result (total size: (8930, 27)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-140.879656,4.0135e-13,1.7289e-10,16.655402,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,1.7289e-10,"""P08865""",1.3443e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116"""
"""KQVEQLEK""",-15.259082,7.629541,-140.760114,4.0369e-13,1.7289e-10,16.654236,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,1.7289e-10,"""Q14152""",2.8258e-12,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680"""
"""ADHQPLTEA""",-14.459833,7.229916,-138.633016,4.4808e-13,1.7289e-10,16.63305,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,1.7289e-10,"""P08865""",1.3443e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138"""
"""ATSISPEQCIK""",-14.37804,7.18902,-136.600459,4.9581e-13,1.7289e-10,16.611987,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,1.7289e-10,"""O75122""",4.9581e-13,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182"""
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-131.412513,6.4645e-13,1.7289e-10,16.554311,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,1.7289e-10,"""Q9NPI6""",1.9394e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477"""


#### Aggregating Test Results from Peptides to Cut Sites

- Here, we use the `"combine_p_direction_check"` pipeline as an example.


In [30]:
stats_cut_site_agg = lipana.stats.pipe.do_stats_pipe_agg(
    qdf=stats_semi_pep,
    base_entry="stripped_peptide",
    target_entry="cut_site",
    group_entry="protein_group",
    annotation_df=report.df.filter(~pl.col("cut_site_is_restricted")),
    annotation_cols=[
        "cut_site",
        "cut_site_n_aa",
        "cut_site_c_aa",
        "cut_site_on_term",
        "cut_site_is_restricted",
    ],
    pipeline="combine_p_direction_check",
)
print(f"First 5 rows of the result (total size: {stats_cut_site_agg.shape}):")
stats_cut_site_agg.head(5)

/home/lishsh/LiPAna_new_version/lipana/stats/infer.py:101: RuntimeWarning: All-NaN slice encountered
  func_out = func(x)


First 5 rows of the result (total size: (8998, 43)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,cut_site,cut_site_n_aa,cut_site_c_aa,cut_site_on_term,cut_site_is_restricted,row_sign,n_detection_in_group,is_group_sign_balanced,group_major_sign,sign_filter_passed,log2_fc_combined,log2_fc_limma_combined,pvalue_combined,first_nonnan_in_combined,adj_pvalue_combined_exp_wise,adj_pvalue_combined_group_wise
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,f64,u32,bool,str,bool,f32,f64,f64,bool,f64,f64
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-140.879656,4.0135e-13,1.7289e-10,16.655402,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,1.7289e-10,"""P08865""",1.3443e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""","""P08865-115_116""","""A""","""F""","""c""",false,-1.0,1,false,"""neg""",true,-15.044158,-15.043514,4.0135e-13,true,6.7281e-11,1.3443e-12
"""KQVEQLEK""",-15.259082,7.629541,-140.760114,4.0369e-13,1.7289e-10,16.654236,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,1.7289e-10,"""Q14152""",2.8258e-12,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""","""Q14152-671_672""","""A""","""K""","""n""",false,-1.0,1,false,"""neg""",true,-15.260145,-15.259082,4.0369e-13,true,6.7281e-11,2.8258e-12
"""ADHQPLTEA""",-14.459833,7.229916,-138.633016,4.4808e-13,1.7289e-10,16.63305,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,1.7289e-10,"""P08865""",1.3443e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""","""P08865-137_138""","""A""","""S""","""c""",false,-1.0,2,false,"""neg""",true,-14.459851,-14.459833,4.4808e-13,true,6.7281e-11,1.3443e-12
"""ATSISPEQCIK""",-14.37804,7.18902,-136.600459,4.9581e-13,1.7289e-10,16.611987,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,1.7289e-10,"""O75122""",4.9581e-13,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""","""O75122-1170_1171""","""L""","""A""","""n""",false,-1.0,1,false,"""neg""",true,-14.378295,-14.37804,4.9581e-13,true,6.7281e-11,4.9581e-13
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-131.412513,6.4645e-13,1.7289e-10,16.554311,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,1.7289e-10,"""Q9NPI6""",1.9394e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477""","""Q9NPI6-463_464""","""M""","""Q""","""n""",false,-1.0,1,false,"""neg""",true,-14.077208,-14.076481,6.4645e-13,true,7.9749e-11,1.9394e-12


#### Selecting Significant Cut Sites

- This example continues to use the specific rules for this dataset. The difference compared to selecting fully-specific peptides in Example 1 lies in the columns used: `"adj_pvalue_combined_group_wise"` and `"log2_fc_limma_combined"`. These two columns are generated specifically by the `"combine_p"` aggregation pipeline.
- Note: To retain peptide-level information, you might observe duplicated rows for some cut sites within each pair, as a single cut site can originate from multiple peptides. The statistical results will be identical for each instance of a cut site within a pair. You can simply select unique entries based on the combination of `("cut_site", "pair")` columns if the peptide-level information is not required.


In [31]:
stats_cut_site_agg = lipana.stats.do_significant_selection(
    stats_cut_site_agg,
    rules=[
        lipana.stats.SignificantRule(
            greater_than=None,
            less_than=(("adj_pvalue_combined_group_wise", 0.05), ("log2_fc_limma_combined", -np.log2(1.5))),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=pl.col("pair").eq("H2_Y100_vs_H5_Y100"),
        ),
        lipana.stats.SignificantRule(
            greater_than=("log2_fc_limma_combined", np.log2(1.5)),
            less_than=("adj_pvalue_combined_group_wise", 0.05),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=pl.col("pair").eq("H30_Y100_vs_H5_Y100"),
        ),
    ],
    requires_n_passed=1,
    target=None,
    mark_col="significant",
)
filtered = stats_cut_site_agg.filter(pl.col("significant"))
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (1277, 44)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,cut_site,cut_site_n_aa,cut_site_c_aa,cut_site_on_term,cut_site_is_restricted,row_sign,n_detection_in_group,is_group_sign_balanced,group_major_sign,sign_filter_passed,log2_fc_combined,log2_fc_limma_combined,pvalue_combined,first_nonnan_in_combined,adj_pvalue_combined_exp_wise,adj_pvalue_combined_group_wise,significant
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,f64,u32,bool,str,bool,f32,f64,f64,bool,f64,f64,bool
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-140.879656,4.0135e-13,1.7289e-10,16.655402,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,1.7289e-10,"""P08865""",1.3443e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""","""P08865-115_116""","""A""","""F""","""c""",false,-1.0,1,false,"""neg""",true,-15.044158,-15.043514,4.0135e-13,true,6.7281e-11,1.3443e-12,true
"""KQVEQLEK""",-15.259082,7.629541,-140.760114,4.0369e-13,1.7289e-10,16.654236,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,1.7289e-10,"""Q14152""",2.8258e-12,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""","""Q14152-671_672""","""A""","""K""","""n""",false,-1.0,1,false,"""neg""",true,-15.260145,-15.259082,4.0369e-13,true,6.7281e-11,2.8258e-12,true
"""ADHQPLTEA""",-14.459833,7.229916,-138.633016,4.4808e-13,1.7289e-10,16.63305,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,1.7289e-10,"""P08865""",1.3443e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""","""P08865-137_138""","""A""","""S""","""c""",false,-1.0,2,false,"""neg""",true,-14.459851,-14.459833,4.4808e-13,true,6.7281e-11,1.3443e-12,true
"""ATSISPEQCIK""",-14.37804,7.18902,-136.600459,4.9581e-13,1.7289e-10,16.611987,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,1.7289e-10,"""O75122""",4.9581e-13,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""","""O75122-1170_1171""","""L""","""A""","""n""",false,-1.0,1,false,"""neg""",true,-14.378295,-14.37804,4.9581e-13,true,6.7281e-11,4.9581e-13,true
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-131.412513,6.4645e-13,1.7289e-10,16.554311,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,1.7289e-10,"""Q9NPI6""",1.9394e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477""","""Q9NPI6-463_464""","""M""","""Q""","""n""",false,-1.0,1,false,"""neg""",true,-14.077208,-14.076481,6.4645e-13,true,7.9749e-11,1.9394e-12,true


### Example 3: Test on Cut Site Level Directly

This example demonstrates a single-step process for performing tests directly at the cut site level, using the cut-site-level quantification matrix generated previously.

#### Performing Hypothesis Tests on Cut Sites

- Hypothesis tests are performed directly on the pre-aggregated cut-site-level quantification data.
- You will notice that the `annotation_cols` parameter might have commented-out entries corresponding to columns used in Example 2. These columns are excluded here because they relate to the peptide origin of each cut site (peptide-related information) and could lead to duplicate rows (e.g., a cut site might originate from the N-terminus of one peptide and the C-terminus of another).


In [32]:
design = lipana.ComparisonDesign(exp_layout, (("H2_Y100", "H5_Y100"), ("H30_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = (
    (pl.col("peptide_enzymatic_specificity") == "semi_specific")
    & (~pl.col("cut_site_is_restricted"))
    & (pl.col("mapped_species_from_peptide").is_in(("HUMAN", "YEAST")))
)

stats_cut_site_direct = lipana.stats.pipe.do_stats_pipe_direct(
    qdf=report.get_quant_data(
        entry_name="cut_site",
        quant_name="nonrestricted_cut_site_by_maxlfq_from_prec",
        main_df_entry_filter=filters,
    ),
    design=design,
    entry_level="cut_site",
    mv_config=lipana.FullEmptyFillingMissingValueHandler(3),
    min_rep_count=3,
    annotation_df=report.df.filter(filters),
    group_entry_level="protein_group",
    annotation_cols=[
        "mapped_species_from_peptide",
        # "cut_site_n_aa",
        # "cut_site_c_aa",
        # "cut_site_on_term",
        "cut_site_is_restricted",
    ],
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats_cut_site_direct.shape}):")
stats_cut_site_direct.head(5)

design.pairwise_comparisons = (('H2_Y100', 'H5_Y100'), ('H30_Y100', 'H5_Y100'))


Quantification file: /tmp/tmpren643sr.txt
All defined comparison pairs: c("H2_Y100", "H5_Y100")
Available comparison pairs: c("H2_Y100", "H5_Y100")
Comparing H2_Y100 vs H5_Y100
Output to /tmp/tmpren643sr.txt-limma_output.txt
Quantification file: /tmp/tmpxd02gea1.txt
All defined comparison pairs: c("H30_Y100", "H5_Y100")
Available comparison pairs: c("H30_Y100", "H5_Y100")
Comparing H30_Y100 vs H5_Y100
Output to /tmp/tmpxd02gea1.txt-limma_output.txt


First 5 rows of the result (total size: (8204, 18)):


cut_site,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,mapped_species_from_peptide,cut_site_is_restricted
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,str,bool
"""P08865-115_116""",-15.043514,7.521757,-143.7594,3.7243e-13,1.6529e-10,16.735576,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,1.6529e-10,"""P08865""",2.2320e-12,"""HUMAN""",false
"""Q14152-671_672""",-15.259082,7.629541,-143.543594,3.7628e-13,1.6529e-10,16.733524,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,1.6529e-10,"""Q14152""",3.0102e-12,"""HUMAN""",false
"""O75122-1170_1171""",-14.37804,7.18902,-139.482463,4.5782e-13,1.6529e-10,16.693329,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,1.6529e-10,"""O75122""",4.5782e-13,"""HUMAN""",false
"""Q9NPI6-463_464""",-14.076481,7.03824,-134.080362,5.9970e-13,1.6529e-10,16.63484,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,1.6529e-10,"""Q9NPI6""",1.7991e-12,"""HUMAN""",false
"""O43809-27_28""",-14.146611,7.073305,-132.069392,6.6495e-13,1.6529e-10,16.611459,"""H2_Y100_vs_H5_Y100""",-14.147872,0,3,"""full_empty""",true,1.6529e-10,"""O43809""",6.6495e-13,"""HUMAN""",false


#### Selecting Significant Cut Sites

- This example performs the selection of significant detections based on the direct test results, similar to Example 1. It does not require modifying the columns used for significance criteria, unlike the aggregation approach in Example 2.


In [33]:
stats_cut_site_direct = lipana.stats.do_significant_selection(
    stats_cut_site_direct,
    rules=[
        lipana.stats.SignificantRule(
            greater_than=None,
            less_than=(("adj_pvalue_group_wise", 0.05), ("log2_fc_limma", -np.log2(1.5))),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=pl.col("pair").eq("H2_Y100_vs_H5_Y100"),
        ),
        lipana.stats.SignificantRule(
            greater_than=("log2_fc_limma", np.log2(1.5)),
            less_than=("adj_pvalue_group_wise", 0.05),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=pl.col("pair").eq("H30_Y100_vs_H5_Y100"),
        ),
    ],
    requires_n_passed=1,
    target=None,
    mark_col="significant",
)
filtered = stats_cut_site_direct.filter(pl.col("significant"))
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (1243, 19)):


cut_site,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,mapped_species_from_peptide,cut_site_is_restricted,significant
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,str,bool,bool
"""P08865-115_116""",-15.043514,7.521757,-143.7594,3.7243e-13,1.6529e-10,16.735576,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,1.6529e-10,"""P08865""",2.2320e-12,"""HUMAN""",false,true
"""Q14152-671_672""",-15.259082,7.629541,-143.543594,3.7628e-13,1.6529e-10,16.733524,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,1.6529e-10,"""Q14152""",3.0102e-12,"""HUMAN""",false,true
"""O75122-1170_1171""",-14.37804,7.18902,-139.482463,4.5782e-13,1.6529e-10,16.693329,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,1.6529e-10,"""O75122""",4.5782e-13,"""HUMAN""",false,true
"""Q9NPI6-463_464""",-14.076481,7.03824,-134.080362,5.9970e-13,1.6529e-10,16.63484,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,1.6529e-10,"""Q9NPI6""",1.7991e-12,"""HUMAN""",false,true
"""O43809-27_28""",-14.146611,7.073305,-132.069392,6.6495e-13,1.6529e-10,16.611459,"""H2_Y100_vs_H5_Y100""",-14.147872,0,3,"""full_empty""",true,1.6529e-10,"""O43809""",6.6495e-13,"""HUMAN""",false,true


### Example 4: Test on All Stripped Peptides and Aggregate to Protein Group Level

This example performs hypothesis tests on all stripped peptides (regardless of specificity) and then aggregates the results to the protein group level. The aggregation method used here selects the base peptide(s) with the minimum p-value(s) for each protein group (e.g., using `"sel_min_p"` or `"sel_min2_p"`).

#### Performing Hypothesis Tests on All Peptides


In [34]:
design = lipana.ComparisonDesign(exp_layout, (("H2_Y100", "H5_Y100"), ("H30_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = pl.col("mapped_species_from_peptide").is_in(("HUMAN", "YEAST"))

stats_all_pep = lipana.stats.pipe.do_stats_pipe_direct(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_ms1ms2",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    design=design,
    entry_level="stripped_peptide",
    mv_config=lipana.FullEmptyFillingMissingValueHandler(3),
    min_rep_count=3,
    annotation_df=report.df.filter(filters),
    group_entry_level="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    annotation_cols=[
        "peptide_start_position",
        "peptide_end_position",
        "prev_aa",
        "next_aa",
        "peptide_n_term_aa",
        "peptide_c_term_aa",
        "nterm_enzymatic_specificity",
        "cterm_enzymatic_specificity",
        "peptide_enzymatic_specificity",
        "n_cut_site",
        "c_cut_site",
    ],
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats_all_pep.shape}):")
stats_all_pep.head(5)

design.pairwise_comparisons = (('H2_Y100', 'H5_Y100'), ('H30_Y100', 'H5_Y100'))


Quantification file: /tmp/tmp_sxercf6.txt
All defined comparison pairs: c("H2_Y100", "H5_Y100")
Available comparison pairs: c("H2_Y100", "H5_Y100")
Comparing H2_Y100 vs H5_Y100
Output to /tmp/tmp_sxercf6.txt-limma_output.txt
Quantification file: /tmp/tmpj4g36a2n.txt
All defined comparison pairs: c("H30_Y100", "H5_Y100")
Available comparison pairs: c("H30_Y100", "H5_Y100")
Comparing H30_Y100 vs H5_Y100
Output to /tmp/tmpj4g36a2n.txt-limma_output.txt


First 5 rows of the result (total size: (19056, 27)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-150.948364,5.8799e-13,3.0867e-10,16.290224,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116"""
"""KQVEQLEK""",-15.259082,7.629541,-150.398987,6.0235e-13,3.0867e-10,16.285656,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,3.0867e-10,"""Q14152""",2.1684e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680"""
"""ADHQPLTEA""",-14.459833,7.229916,-149.22073,6.3453e-13,3.0867e-10,16.275707,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138"""
"""ATSISPEQCIK""",-14.37804,7.18902,-146.76885,7.0805e-13,3.0867e-10,16.25432,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,3.0867e-10,"""O75122""",1.4161e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182"""
"""RIEYIEAR""",-14.269712,7.134856,-142.327075,8.6772e-13,3.0867e-10,16.213065,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.0867e-10,"""Q8WUW1""",8.6772e-13,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68"""


#### Selecting Significant Protein Groups Based on the Minimum P-value Peptide

- Note: In this pipeline, a `top_k_selected` column is added to the results. This column indicates whether a peptide is among the top *k* peptides with the lowest p-values within its group (here, *k*=1). In the `do_significant_selection` function, an additional condition `pl.col("top_k_selected")` is added to the `filter_condition` parameter. This ensures that the significance selection is based only on the single peptide with the minimum p-value within each protein group.


In [35]:
stats_protein_group_agg_pep_minp = lipana.stats.pipe.do_stats_pipe_agg(
    qdf=stats_all_pep,
    base_entry="stripped_peptide",
    target_entry="protein_group",
    group_entry=None,
    annotation_df=None,
    annotation_cols=None,
    pipeline="sel_min_p",
)
print(f"First 5 rows of the result (total size: {stats_protein_group_agg_pep_minp.shape}):")
stats_protein_group_agg_pep_minp.head(5)

First 5 rows of the result (total size: (19056, 28)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-150.948364,5.8799e-13,3.0867e-10,16.290224,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""",true
"""KQVEQLEK""",-15.259082,7.629541,-150.398987,6.0235e-13,3.0867e-10,16.285656,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,3.0867e-10,"""Q14152""",2.1684e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""",true
"""ADHQPLTEA""",-14.459833,7.229916,-149.22073,6.3453e-13,3.0867e-10,16.275707,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,NaN,"""P08865""",3.4899e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""",false
"""ATSISPEQCIK""",-14.37804,7.18902,-146.76885,7.0805e-13,3.0867e-10,16.25432,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,3.0867e-10,"""O75122""",1.4161e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""",true
"""RIEYIEAR""",-14.269712,7.134856,-142.327075,8.6772e-13,3.0867e-10,16.213065,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.0867e-10,"""Q8WUW1""",8.6772e-13,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""",true


In [36]:
stats_protein_group_agg_pep_minp = lipana.stats.do_significant_selection(
    stats_protein_group_agg_pep_minp,
    rules=[
        lipana.stats.SignificantRule(
            greater_than=None,
            less_than=(("adj_pvalue_group_wise", 0.05), ("log2_fc_limma", -np.log2(1.5))),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=(pl.col("pair").eq("H2_Y100_vs_H5_Y100") & pl.col("top_k_selected")),
        ),
        lipana.stats.SignificantRule(
            greater_than=("log2_fc_limma", np.log2(1.5)),
            less_than=("adj_pvalue_group_wise", 0.05),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=(pl.col("pair").eq("H30_Y100_vs_H5_Y100") & pl.col("top_k_selected")),
        ),
    ],
    requires_n_passed=1,
    target=None,
    mark_col="significant",
)
filtered = stats_protein_group_agg_pep_minp.filter(pl.col("significant"))
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (788, 29)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected,significant
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool,bool
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-150.948364,5.8799e-13,3.0867e-10,16.290224,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""",true,true
"""KQVEQLEK""",-15.259082,7.629541,-150.398987,6.0235e-13,3.0867e-10,16.285656,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,3.0867e-10,"""Q14152""",2.1684e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""",true,true
"""ATSISPEQCIK""",-14.37804,7.18902,-146.76885,7.0805e-13,3.0867e-10,16.25432,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,3.0867e-10,"""O75122""",1.4161e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""",true,true
"""RIEYIEAR""",-14.269712,7.134856,-142.327075,8.6772e-13,3.0867e-10,16.213065,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.0867e-10,"""Q8WUW1""",8.6772e-13,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""",true,true
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-140.721878,9.3535e-13,3.0867e-10,16.197305,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,3.0867e-10,"""Q9NPI6""",2.8061e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477""",true,true


#### Selecting Significant Protein Groups Based on the Top 2 Lowest P-value Peptides

- The primary output here, `stats_protein_group_agg_pep_minp2`, will be nearly identical to the previous output (`stats_protein_group_agg_pep_minp`), except for the values in the `top_k_selected` column. This column will now contain `True` if the peptide is one of the two peptides with the lowest p-values within its protein group.
- The command for selecting significant detections includes the following changes:
  - `requires_n_passed=2`
  - `target="protein_group"`
- After the selection annotation step, in addition to the `significant` column (indicating if an individual peptide meets the significance rule), a `significant_2passed` column will be added. This column annotates whether the protein group has a sufficient number (2 in this case) of significant peptides among the top 2 selected. If you set *k* to 3, this column would be named `significant_3passed`.
- Note: In this scenario, the `rules` parameter for the `do_significant_selection` function should include rules defined with an additional filter condition involving `pl.col("top_k_selected")`. For example: `filter_condition=(pl.col("pair").eq("H30_Y100_vs_H5_Y100") & pl.col("top_k_selected"))`. This setup ensures that we specifically check if the *two peptides with the lowest p-values* are significant, rather than checking if *any* two peptides within the protein group pass the significance criteria.


In [37]:
stats_protein_group_agg_pep_minp2 = lipana.stats.pipe.do_stats_pipe_agg(
    qdf=stats_all_pep,
    base_entry="stripped_peptide",
    target_entry="protein_group",
    group_entry=None,
    annotation_df=None,
    annotation_cols=None,
    pipeline="sel_min2_p",
)
print(f"First 5 rows of the result (total size: {stats_protein_group_agg_pep_minp2.shape}):")
stats_protein_group_agg_pep_minp2.head(5)

First 5 rows of the result (total size: (19056, 28)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-150.948364,5.8799e-13,3.0867e-10,16.290224,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""",true
"""KQVEQLEK""",-15.259082,7.629541,-150.398987,6.0235e-13,3.0867e-10,16.285656,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,3.0867e-10,"""Q14152""",2.1684e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""",true
"""ADHQPLTEA""",-14.459833,7.229916,-149.22073,6.3453e-13,3.0867e-10,16.275707,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""",true
"""ATSISPEQCIK""",-14.37804,7.18902,-146.76885,7.0805e-13,3.0867e-10,16.25432,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,3.0867e-10,"""O75122""",1.4161e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""",true
"""RIEYIEAR""",-14.269712,7.134856,-142.327075,8.6772e-13,3.0867e-10,16.213065,"""H2_Y100_vs_H5_Y100""",-14.270495,0,3,"""full_empty""",true,3.0867e-10,"""Q8WUW1""",8.6772e-13,60,67,"""R""","""V""","""R""","""R""","""restricted""","""restricted""","""fully_specific""","""Q8WUW1-59_60""","""Q8WUW1-67_68""",true


In [38]:
stats_protein_group_agg_pep_minp2 = lipana.stats.do_significant_selection(
    stats_protein_group_agg_pep_minp2,
    rules=[
        lipana.stats.SignificantRule(
            greater_than=None,
            less_than=(("adj_pvalue_exp_wise", 0.05), ("log2_fc_limma", -np.log2(1.5))),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=(pl.col("pair").eq("H2_Y100_vs_H5_Y100") & pl.col("top_k_selected")),
        ),
        lipana.stats.SignificantRule(
            greater_than=("log2_fc_limma", np.log2(1.5)),
            less_than=("adj_pvalue_exp_wise", 0.05),
            gt_value_or_lt_negate=None,
            equal_to=None,
            filter_condition=(pl.col("pair").eq("H30_Y100_vs_H5_Y100") & pl.col("top_k_selected")),
        ),
    ],
    requires_n_passed=2,  # Note here the number of passed rules is required to be 2, which just matches the number of selected top k (minimum p-value) peptides in column `top_k_selected`
    target=(
        "protein_group",
        "pair",
    ),  # Note here the target should be given to concretely specify from where to ensure two peptides are significant
    mark_col="significant",
)
filtered = stats_protein_group_agg_pep_minp2.filter(
    pl.col("significant_2passed")
)  # Note here the selection is based on `significant_2passed` column
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (8391, 30)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected,significant,significant_2passed
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool,bool,bool
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-150.948364,5.8799e-13,3.0867e-10,16.290224,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""",true,true,true
"""KQVEQLEK""",-15.259082,7.629541,-150.398987,6.0235e-13,3.0867e-10,16.285656,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,3.0867e-10,"""Q14152""",2.1684e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""",true,true,true
"""ADHQPLTEA""",-14.459833,7.229916,-149.22073,6.3453e-13,3.0867e-10,16.275707,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""",true,true,true
"""ATSISPEQCIK""",-14.37804,7.18902,-146.76885,7.0805e-13,3.0867e-10,16.25432,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,3.0867e-10,"""O75122""",1.4161e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""",true,true,true
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-140.721878,9.3535e-13,3.0867e-10,16.197305,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,3.0867e-10,"""Q9NPI6""",2.8061e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477""",true,true,true


- Note: in this case, after filtering the results by column "significant_2passed", you will always see all peptides of the selected significant protein groups. If you want to only see those peptides that are selected to support the protein groups that are significant, should add an additional condition to ensure column "significant" is true. That is, the filtered dataframe will have 2n rows to present 2n significant "peptide"+"comparison_pair", in which there are n selected significant "protein"+"comparison_pair"


In [39]:
filtered = stats_protein_group_agg_pep_minp2.filter(pl.col("significant_2passed") & pl.col("significant"))
print(f"First 5 rows of the result (total size: {filtered.shape}):")
filtered.head(5)

First 5 rows of the result (total size: (976, 30)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected,significant,significant_2passed
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool,bool,bool
"""FTPGTFTNQIQAA""",-15.043514,7.521757,-150.948364,5.8799e-13,3.0867e-10,16.290224,"""H2_Y100_vs_H5_Y100""",-15.044158,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,103,115,"""R""","""F""","""F""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-102_103""","""P08865-115_116""",true,true,true
"""KQVEQLEK""",-15.259082,7.629541,-150.398987,6.0235e-13,3.0867e-10,16.285656,"""H2_Y100_vs_H5_Y100""",-15.260145,0,3,"""full_empty""",true,3.0867e-10,"""Q14152""",2.1684e-11,672,679,"""A""","""E""","""K""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q14152-671_672""","""Q14152-679_680""",true,true,true
"""ADHQPLTEA""",-14.459833,7.229916,-149.22073,6.3453e-13,3.0867e-10,16.275707,"""H2_Y100_vs_H5_Y100""",-14.459851,0,3,"""full_empty""",true,3.0867e-10,"""P08865""",3.4899e-12,129,137,"""R""","""S""","""A""","""A""","""restricted""","""non_restricted""","""semi_specific""","""P08865-128_129""","""P08865-137_138""",true,true,true
"""ATSISPEQCIK""",-14.37804,7.18902,-146.76885,7.0805e-13,3.0867e-10,16.25432,"""H2_Y100_vs_H5_Y100""",-14.378295,0,3,"""full_empty""",true,3.0867e-10,"""O75122""",1.4161e-12,1171,1181,"""L""","""V""","""A""","""K""","""non_restricted""","""restricted""","""semi_specific""","""O75122-1170_1171""","""O75122-1181_1182""",true,true,true
"""QQNQDPEVFVQPK""",-14.076481,7.03824,-140.721878,9.3535e-13,3.0867e-10,16.197305,"""H2_Y100_vs_H5_Y100""",-14.077208,0,3,"""full_empty""",true,3.0867e-10,"""Q9NPI6""",2.8061e-12,464,476,"""M""","""V""","""Q""","""K""","""non_restricted""","""restricted""","""semi_specific""","""Q9NPI6-463_464""","""Q9NPI6-476_477""",true,true,true


## Utilities

### Retrieve annotation from main report for quantification data or statistic result

- To reduce the additional use of memory, all annotations are generally only stored in the report
- To retrieve these information from the main report, the table join can be done on the dataframe that needs some information and the main report `report.df`
- To handle some potential complicated cases, there is also a function `lipana.attach_annotation_from_other_df` to do this action
- Note: It is highly recommended to attach the desired annotations to the quantification dataframe and thoroughly verify their accuracy before proceeding with statistical analysis. This verification step is crucial because annotations can sometimes exhibit one-to-many (1:n) mappings. For example, reports from analysis software might list the same peptide sequence on multiple rows, associating it with different protein groups or genes in each instance.

In [40]:
# Retrieve some protein annotations for the stats result
stats_protein_group_agg_pep_minp = lipana.attach_annotation_from_other_df(
    df=stats_protein_group_agg_pep_minp,
    other_df=report.df,
    annotation_cols=[
        "Protein.Names",
        "Genes",
        "First.Protein.Description",
    ],
    on="protein_group",
    pre_filter=None,
    unique_on_key_only=False,
    method="leftjoin",
)
stats_protein_group_agg_pep_minp.sort("protein_group").unique("protein_group").head(5)

stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,detected_runs-numerator,detected_runs-denominator,missing_fill_type,mv_check_passed,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,peptide_start_position,peptide_end_position,prev_aa,next_aa,peptide_n_term_aa,peptide_c_term_aa,nterm_enzymatic_specificity,cterm_enzymatic_specificity,peptide_enzymatic_specificity,n_cut_site,c_cut_site,top_k_selected,significant,Protein.Names,Genes,First.Protein.Description
str,f64,f64,f64,f64,f64,f64,str,f32,i8,i8,str,bool,f64,str,f64,u32,i64,str,str,str,str,str,str,str,str,str,bool,bool,str,str,str
"""AVTEQGAELSNEER""",-1.265568,14.462931,-10.838603,0.000019,0.000204,3.213971,"""H2_Y100_vs_H5_Y100""",-1.266182,3,3,"""""",true,0.000204,"""P27348""",0.000081,28,41,"""K""","""N""","""A""","""R""","""restricted""","""restricted""","""fully_specific""","""P27348-27_28""","""P27348-41_42""",true,true,"""1433T_HUMAN""","""YWHAQ""","""14-3-3 protein theta"""
"""DDVDDFQPR""",null,null,null,null,null,null,"""H2_Y100_vs_H5_Y100""",-0.103635,3,2,"""""",false,NaN,"""Q08816""",NaN,92,100,"""L""","""I""","""D""","""R""","""non_restricted""","""restricted""","""semi_specific""","""Q08816-91_92""","""Q08816-100_101""",false,false,"""YO352_YEAST""","""YOR352W""","""Uncharacterized protein YOR352…"
"""CVSSPHFQVAER""",-12.845214,6.422607,-94.440506,1.3089e-11,5.1422e-10,15.425704,"""H2_Y100_vs_H5_Y100""",-12.855769,0,3,"""full_empty""",true,5.1422e-10,"""Q14738""",2.6178e-11,410,421,"""K""","""A""","""C""","""R""","""restricted""","""restricted""","""fully_specific""","""Q14738-409_410""","""Q14738-421_422""",true,true,"""2A5D_HUMAN""","""PPP2R5D""","""Serine/threonine-protein phosp…"
"""EEEEEEEQKIGK""",null,null,null,null,null,null,"""H2_Y100_vs_H5_Y100""",NaN,0,0,"""""",false,NaN,"""P23025""",NaN,78,89,"""L""","""V""","""E""","""K""","""non_restricted""","""restricted""","""semi_specific""","""P23025-77_78""","""P23025-89_90""",false,false,"""XPA_HUMAN""","""XPA""","""DNA repair protein complementi…"
"""GSSSYSQLLAATCLTK""",null,null,null,null,null,null,"""H2_Y100_vs_H5_Y100""",-0.200971,1,1,"""""",false,NaN,"""Q9UIA9""",NaN,54,69,"""R""","""L""","""G""","""K""","""restricted""","""restricted""","""fully_specific""","""Q9UIA9-53_54""","""Q9UIA9-69_70""",false,false,"""XPO7_HUMAN""","""XPO7""","""Exportin-7"""
